# ============================================================
# STEP 2D — GRAPH-BASED INFRASTRUCTURE FEATURES (18 features)
# Turkish Quishing Detection v10 — NOVELTY CENTERPIECE
# ============================================================
# 100% OFFLINE — built from domains/tokens you ALREADY have.
# No network. No API. Full coverage on all 1.24M rows.
#
# CORE IDEA: phishing campaigns reuse infrastructure and naming
# patterns. Build a graph where nodes are registered domains and
# edges connect domains that share:
#   - the same TLD + similar token structure
#   - shared rare path/subdomain tokens (campaign fingerprints)
#   - same suspicious-TLD + brand-impersonation pattern
# Malicious domains cluster tightly; benign domains are diffuse.
#
# Graph metrics per domain become features. Evasion-resistant
# because attackers cannot cheaply avoid infrastructure reuse.
#
# Input  : features_full_final.csv  (116 features, after Step 2C)
# Output : features_full_final.csv  (134 features)
#          graph_feature_names_final.pkl
# ============================================================

In [ ]:
import os, re, math, pickle, warnings
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")
SEED = 42

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/"
INPUT_FILE  = os.path.join(DATA_DIR, "features_full_final.csv")
OUTPUT_CSV  = os.path.join(DATA_DIR, "features_full_final.csv")
OUTPUT_PKL  = os.path.join(DATA_DIR, "graph_feature_names_final.pkl")

GRAPH_FEATURES = [
    # Token co-occurrence (campaign fingerprinting) — 6
    "rare_token_count","max_token_cluster_size","shared_token_degree",
    "campaign_token_score","unique_token_ratio","token_reuse_score",
    # Domain structure clustering — 5
    "domain_family_size","tld_token_cooccur","sibling_domain_count",
    "domain_ngram_cluster","registrant_pattern_score",
    # Infrastructure reuse — 4
    "subdomain_reuse_count","path_template_reuse","host_pattern_degree",
    "campaign_membership",
    # Centrality / position — 3
    "token_centrality","is_hub_domain","cluster_malicious_ratio",
]
assert len(GRAPH_FEATURES) == 18

def tokenize_domain(domain):
    return [t for t in re.split(r"[.\-_]", str(domain).lower()) if len(t) >= 3]

def tokenize_url_struct(url):
    """Extract structural tokens: subdomain parts, path segments."""
    u = str(url).lower()
    u = re.sub(r"^https?://", "", u)
    parts = re.split(r"[/.\-_?=&]", u)
    return [p for p in parts if len(p) >= 4 and not p.isdigit()]

print("=" * 70)
print("STEP 2D — GRAPH-BASED INFRASTRUCTURE FEATURES (18)")
print("=" * 70)

df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"\n[1] Loaded {len(df):,} rows")
n = len(df)

urls    = df["url"].astype(str).values
domains = df["reg_domain"].astype(str).values if "reg_domain" in df.columns \
          else np.array([""] * n)
tlds    = df["tld"].astype(str).values if "tld" in df.columns else np.array([""]*n)
labels  = df["label_enc"].values if "label_enc" in df.columns else np.zeros(n)

# ════════════════════════════════════════════════════════════
# PASS 1 — BUILD GLOBAL FREQUENCY TABLES (the "graph")
# ════════════════════════════════════════════════════════════
print(f"\n[2] Pass 1: building token co-occurrence tables ...")

token_domain_count = Counter()      # token -> # distinct domains using it
token_total_count  = Counter()      # token -> total occurrences
token_malicious    = defaultdict(int)  # token -> count in malicious
token_benign       = defaultdict(int)
domain_tokens      = {}             # domain -> set of structural tokens
tld_token_pairs    = Counter()      # (tld, token) -> count
domain_family      = Counter()      # domain n-gram signature -> count

seen_domain = set()
for i in range(n):
    toks = set(tokenize_url_struct(urls[i]))
    domain_tokens[i] = toks
    for t in toks:
        token_total_count[t] += 1
        if labels[i] == 1: token_malicious[t] += 1
        else:              token_benign[t] += 1
        tld_token_pairs[(tlds[i], t)] += 1
    # domain family = sorted char-trigram signature of domain
    d = domains[i]
    if d and d not in seen_domain:
        seen_domain.add(d)
        dtoks = tokenize_domain(d)
        for t in dtoks:
            token_domain_count[t] += 1
    # domain family signature (first 3 + last 3 chars)
    if len(d) >= 6:
        sig = d[:3] + d[-3:]
        domain_family[sig] += 1

    if (i + 1) % 200_000 == 0:
        print(f"    pass1 {i+1:,}/{n:,}")

# Identify rare tokens (campaign fingerprints): appear in few domains
# but multiple URLs = reused within a campaign
RARE_THRESHOLD = 50      # token in fewer than N distinct domains
CAMPAIGN_MIN   = 3       # but appears >= 3 times total

print(f"    Unique structural tokens: {len(token_total_count):,}")
print(f"    Unique domain families:   {len(domain_family):,}")

# ════════════════════════════════════════════════════════════
# PASS 2 — COMPUTE PER-URL GRAPH FEATURES
# ════════════════════════════════════════════════════════════
print(f"\n[3] Pass 2: computing per-URL graph features ...")

records = []
for i in range(n):
    toks = domain_tokens[i]
    d    = domains[i]
    tld  = tlds[i]
    f = {}

    # ── Token co-occurrence / campaign fingerprinting ────────
    rare = [t for t in toks
            if token_total_count[t] >= CAMPAIGN_MIN
            and token_domain_count.get(t, 999) < RARE_THRESHOLD]
    f["rare_token_count"] = len(rare)
    # largest cluster: max # URLs sharing any one of this URL's tokens
    f["max_token_cluster_size"] = max(
        (token_total_count[t] for t in toks), default=0)
    # shared-token degree: sum of co-occurrence across tokens
    f["shared_token_degree"] = sum(token_total_count[t] for t in toks)
    # campaign score: tokens that are rare AND skewed malicious
    camp = 0
    for t in rare:
        mal = token_malicious[t]; ben = token_benign[t]
        if mal + ben > 0 and mal / (mal + ben) > 0.8:
            camp += 1
    f["campaign_token_score"] = camp
    nt = max(len(toks), 1)
    uniq = sum(1 for t in toks if token_total_count[t] <= 2)
    f["unique_token_ratio"] = uniq / nt
    f["token_reuse_score"]  = sum(
        1 for t in toks if token_total_count[t] > 10) / nt

    # ── Domain structure clustering ──────────────────────────
    sig = d[:3] + d[-3:] if len(d) >= 6 else d
    f["domain_family_size"] = domain_family.get(sig, 1)
    f["tld_token_cooccur"]  = sum(
        tld_token_pairs.get((tld, t), 0) for t in toks)
    # sibling domains: same family signature
    f["sibling_domain_count"] = max(domain_family.get(sig, 1) - 1, 0)
    dtoks = tokenize_domain(d)
    f["domain_ngram_cluster"] = sum(
        token_domain_count.get(t, 0) for t in dtoks)
    # registrant pattern proxy: domain length + digit + hyphen signature
    f["registrant_pattern_score"] = (
        len(d) + d.count("-") * 3 + sum(c.isdigit() for c in d) * 2)

    # ── Infrastructure reuse ─────────────────────────────────
    f["subdomain_reuse_count"] = sum(
        1 for t in toks if token_domain_count.get(t, 0) > 5)
    f["path_template_reuse"]   = sum(
        token_total_count[t] for t in rare)
    f["host_pattern_degree"]   = len([
        t for t in toks if token_total_count[t] >= CAMPAIGN_MIN])
    f["campaign_membership"]   = int(camp > 0 or f["domain_family_size"] > 5)

    # ── Centrality / position ────────────────────────────────
    f["token_centrality"] = round(
        f["shared_token_degree"] / nt, 2)
    f["is_hub_domain"]    = int(f["max_token_cluster_size"] > 100)
    # malicious ratio of the tokens this URL contains
    mal_sum = sum(token_malicious[t] for t in toks)
    tot_sum = sum(token_malicious[t] + token_benign[t] for t in toks)
    f["cluster_malicious_ratio"] = round(mal_sum / max(tot_sum, 1), 4)

    records.append({k: f[k] for k in GRAPH_FEATURES})
    if (i + 1) % 200_000 == 0:
        print(f"    pass2 {i+1:,}/{n:,}")

g_df = pd.DataFrame(records)
print(f"\n[4] Graph matrix: {g_df.shape}  Nulls: {g_df.isnull().sum().sum()}")

# ── LEAKAGE GUARD ─────────────────────────────────────────────
# cluster_malicious_ratio uses labels — it is computed GLOBALLY
# across the whole dataset, which leaks test labels into train.
# Provide BOTH versions: keep for analysis, flag for removal.
print(f"\n[5] ⚠ LEAKAGE NOTE:")
print(f"    'cluster_malicious_ratio' and 'campaign_token_score' use")
print(f"    label info computed across the FULL dataset. For honest")
print(f"    CV, recompute these per-fold (train-only) OR drop them.")
print(f"    They are included but flagged in graph_feature_names.pkl.")

LEAKY_GRAPH = ["cluster_malicious_ratio", "campaign_token_score"]

# Merge
META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
EXISTING = [c for c in df.columns if c not in META]
full = pd.concat([df[EXISTING].reset_index(drop=True),
                  g_df.reset_index(drop=True)], axis=1)
for c in META:
    if c in df.columns:
        full[c] = df[c].values
full.to_csv(OUTPUT_CSV, index=False)
with open(OUTPUT_PKL, "wb") as fh:
    pickle.dump({"all": GRAPH_FEATURES, "leaky": LEAKY_GRAPH,
                 "safe": [f for f in GRAPH_FEATURES if f not in LEAKY_GRAPH]}, fh)
print(f"\n[6] Saved {len(EXISTING)+18} features → {OUTPUT_CSV}")

# Class rates
print(f"\n[7] Graph feature rates by class:")
g_df["class_final"] = df["class_final"].values
key = ["campaign_membership","domain_family_size","rare_token_count",
       "is_hub_domain","token_reuse_score","sibling_domain_count"]
print(g_df.groupby("class_final")[key].mean().round(3).to_string())

# MI (excluding leaky features for honest assessment)
print(f"\n[8] MI ranking — SAFE graph features (5-class):")
safe = [f for f in GRAPH_FEATURES if f not in LEAKY_GRAPH]
X = g_df[safe].values
mi = mutual_info_classif(X, df["class_enc"].values, random_state=SEED)
mi_s = pd.Series(mi, index=safe).sort_values(ascending=False)
print(mi_s.head(10).round(4).to_string())

print(f"""
{'='*70}
STEP 2D COMPLETE — 18 graph features (134 total)
{'='*70}
  Novelty centerpiece: campaign-fingerprint graph features
  Top SAFE graph feature: {mi_s.index[0]} (MI={mi_s.iloc[0]:.4f})

  ⚠ 2 features ({', '.join(LEAKY_GRAPH)}) use global label info.
    For Step 3, use the 'safe' list from graph_feature_names.pkl,
    OR recompute them per-fold (train-only statistics).

  Output: {OUTPUT_CSV}
  Feature total: 96 + 20 (Turkish) + 18 (graph) = 134
  Next → Step 3 with the expanded feature set
{'='*70}
""")

STEP 2D — GRAPH-BASED INFRASTRUCTURE FEATURES (18)

[1] Loaded 1,239,308 rows

[2] Pass 1: building token co-occurrence tables ...
    pass1 200,000/1,239,308
    pass1 400,000/1,239,308
    pass1 600,000/1,239,308
    pass1 800,000/1,239,308
    pass1 1,000,000/1,239,308
    pass1 1,200,000/1,239,308
    Unique structural tokens: 634,425
    Unique domain families:   145,086

[3] Pass 2: computing per-URL graph features ...
    pass2 200,000/1,239,308
    pass2 400,000/1,239,308
    pass2 600,000/1,239,308
    pass2 800,000/1,239,308
    pass2 1,000,000/1,239,308
    pass2 1,200,000/1,239,308

[4] Graph matrix: (1239308, 18)  Nulls: 0

[5] ⚠ LEAKAGE NOTE:
    'cluster_malicious_ratio' and 'campaign_token_score' use
    label info computed across the FULL dataset. For honest
    CV, recompute these per-fold (train-only) OR drop them.
    They are included but flagged in graph_feature_names.pkl.

[6] Saved 139 features → /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/fe